# Session 5 Lab — Words as Data, and What a Language Model Measures

Six blocks, in the order of the decks. Each block ends with a **gate**: a bound fixed before the run.
Nothing here needs a network connection or a GPU; from a clean kernel the notebook runs in a few minutes.

| Block | min | You will | Deck |
|---|---|---|---|
| **A** | 20 | build a vocabulary twice — byte-pair merges learned by hand, then TF-IDF — fitted on 1994–2014 and nothing else | 5.1 |
| **B** | 20 | train word vectors on the same window, build a hawk–dove axis, and watch it **fail** validation beside a ten-line rule that passes | 5.1, 5.2 |
| **C** | 15 | do the attention arithmetic by hand, then open Session 1's small transformer: its attention, its writing, its first sixty training steps | 5.2, 5.3 |
| **D** | 20 | read a recorded language-model instrument: provenance, agreement, Cohen's κ, and what convergent validity does not buy | 5.4 |
| **E** | 20 | run chapter 14's four simulations through the chapter's own code: attenuation, correction, per-class κ, self-consistency, order effects | 5.4, 5.5 |
| **F** | 25 | ask whether the Committee's words improve Session 2's bond-yield forecast, under a protocol hashed before the first scored run | 5.6 |

**What a gate is here.** Every gate is on **procedure** or on **reproduction**, never on an outcome.
A procedure gate checks how the run was done: hashes checked before a byte is read, every fitted object
confined to the training window, the attention map causal, the test window scored once. A reproduction
gate checks that a digit some frozen artefact already carries comes back — the corpus manifest, Session 2's
published baselines, chapter 14's printed figures, the recorded run. A reproduction gate has **no preferred
direction**: it would pass exactly as it does now if block B's embedding signal had validated, if block D's
byte-identical repeat had agreed more than the reworded prompt, and if block F's text had improved the
forecast. Nothing in this notebook passes because a number came out in a direction the lab would like.

**One corpus, one window.** All 245 post-meeting FOMC statements, 1994-02-04 to 2026-06-17, hash-checked
against `fomc_manifest.json`. The **training window** is the 149 statements dated on or before 2014-12-31.
Every vocabulary, every IDF weight, every word vector and every scaler in this notebook is fitted on that
window; 2015 onwards is held back and only ever transformed.

The four modules behind the blocks live in `session05_assets/`: `session05_text.py` (A, B),
`session05_attention.py` (C), `session05_measure.py` (E) and `session05_forecast.py` (F). The numbers the
slides quote come from full-budget runs of exactly these functions, stored in `recorded_*.json`.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")       # set before numpy is imported; shared machines need it
%matplotlib inline
import collections, itertools, json, sys, textwrap, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LAB = Path.cwd() if (Path.cwd() / "session05_assets").is_dir() else Path.cwd() / "labs"
ASSETS = LAB / "session05_assets"
sys.path.insert(0, str(ASSETS))
import session05_text as tx            # blocks A, B
import session05_attention as at       # block C
import session05_measure as me         # block E (and block D's kappa)
import session05_forecast as fc        # block F

GATE = json.loads((ASSETS / "lab_gates.json").read_text(encoding="utf-8"))
REC = {k: json.loads((ASSETS / f"recorded_{k}.json").read_text(encoding="utf-8"))
       for k in ("text", "attention", "measure", "forecast")}
TEAL, RED, MUTED = "#006D77", "#A3473B", "#5C686B"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
T_START = time.time()
GATES = []

def gate(name, ok, detail=""):
    GATES.append((name, bool(ok)))
    print(f"GATE {name}: {'pass' if ok else 'FAIL'}   {detail}")
    assert ok, f"gate {name} failed: {detail}"

print("numpy", np.__version__, "| pandas", pd.__version__, "| recorded runs:", ", ".join(REC))

# Block A — Tokens: what the computer actually reads

A statement is a string of characters. No regression takes a string. Between the two sits a **vocabulary**:
a finite list of units, plus a rule that turns any document into counts over that list. Block A builds one
twice — once by merging characters upward (byte-pair encoding) and once by counting words and weighting
them (TF-IDF) — and fits both on the 1994–2014 window alone.

## A1. The corpus, and the line down the middle of it

**Training window**: the statements dated on or before 2014-12-31. **Held back**: everything after.
The line is not decoration. A vocabulary fitted on the whole sample has seen the words of 2020 when it
scores 1994, and every later claim about out-of-sample behaviour is then about a sample that was never out.

In [ ]:
statements = tx.load_corpus()                      # every file's sha256 checked against fomc_manifest.json
train = tx.training_window(statements)             # dated on or before 2014-12-31, and nothing else
held_back = [s for s in statements if s.date > tx.TRAIN_END]

for label, block in (("all", statements), ("training window", train), ("held back", held_back)):
    d = tx.corpus_summary(block)
    print(f"{label:<16} n = {d['n']:>3}   {d['first']} to {d['last']}   {d['characters']:>7,} characters"
          f"   {d['words']:>6,} words   median statement {d['median_characters']:>5,} characters")

per_year = REC["text"]["corpus"]["per_year_1994_1999"]
print("\nstatements per year, 1994-1999:  " + "   ".join(f"{y}: {n}" for y, n in per_year.items()))
print("The Committee met eight times a year throughout. It did not publish a statement after every")
print("meeting until 1999, so the corpus is what was released, not what was decided.")
print(f"\nmedian statement length: {tx.corpus_summary(train)['median_characters']:,} characters in the"
      f" training window against {tx.corpus_summary(held_back)['median_characters']:,} after it —")
print("the object being measured changed shape over the sample, before anyone measured anything.")

ok = (len(statements) == GATE["corpus_statements"]
      and len(train) == GATE["corpus_training_statements"]
      and len(held_back) == GATE["corpus_held_back_statements"]
      and max(s.date for s in train) <= GATE["corpus_training_end"]
      and statements[0].date == GATE["corpus_first"] and statements[-1].date == GATE["corpus_last"])
gate("A1", ok, f"{len(statements)} statements, all hashes match; {len(train)} of them in the window, "
               f"the last dated {max(s.date for s in train)}")

## A2. Five switches, none of them forced by the data

Lower-casing, dropping punctuation, dropping numbers, dropping stop-words, stemming. Each is a defensible
choice; none is implied by the corpus. Every row below is the **same 149 statements** under a different
combination, so the spread across rows is the size of the researcher's hand before a single number is reported.

In [ ]:
ladder = tx.degrees_of_freedom(train)
print(ladder[["tokens", "vocabulary", "vocabulary_vs_baseline", "top_terms"]].to_string())
print(f"\nvocabulary runs from {ladder['vocabulary'].min():,} to {ladder['vocabulary'].max():,} "
      f"— a factor of {ladder['vocabulary'].max() / ladder['vocabulary'].min():.1f} — on one corpus.")
print("the last row's chain:", ladder['steps'].iloc[-1])
print("\nThe stop-word list is short and auditable, and 'not', 'no' and 'less' are deliberately absent:")
print(f"{len(tx.STOPWORDS)} words, and negation carries policy meaning.")

## A3. Byte-pair encoding, learned here

A word-level vocabulary has a hard edge: a word it never saw has no column at all. Byte-pair encoding
removes the edge by building the vocabulary from **below**. Start with characters plus an end-of-word marker;
count adjacent symbol pairs, weighted by word frequency; glue the commonest pair everywhere; repeat.
After 200 rounds the frequent words are single symbols and everything else is spelled out of pieces.

The merges below are learned on the training window. Nothing after 2014 contributed a single count.

In [ ]:
merges = tx.learn_bpe(train, n_merges=GATE["bpe_merges"])       # learned here, from the training window
print(f"{len(merges)} merges, in the order they were learned. The first "
      f"{GATE['bpe_first_merges_checked']}:")
for start in range(0, GATE["bpe_first_merges_checked"], 5):
    print("   " + "   ".join(f"{k + 1:>2}. {a}+{b}"
                             for k, (a, b) in enumerate(merges[start:start + 5], start=start)))
print("\nThe first merges are the grammar of English, discovered from counts: the end-of-word marker glues")
print("onto 'e' and 's' before anything else, and 'th' + 'e</w>' becomes the word 'the' by merge 11.")

In [ ]:
words = tx.word_frequencies(train)                  # the word-level vocabulary of the same window
probes = GATE["bpe_out_of_vocabulary_probes"]
print(f"the word-level vocabulary of the training window: {len(words):,} distinct words\n")
print(f"{'word':<18}{'word-level column?':<22}byte-pair segmentation")
for w in ("raising", "unemployment", "accommodation", *probes):
    pieces = tx.apply_bpe(w, merges)
    print(f"{w:<18}{'yes' if w in words else 'NO — no column at all':<22}" + " | ".join(pieces))
print("\n'reflationary' and 'disinflationary' are real words of monetary economics that this corpus never")
print("uses. A word-level model drops them on the floor. Byte-pair spells them out of pieces it has seen,")
print("and 'infl' and 'tion' carry the parts of the meaning that a count model can still use.")

ok = ([f"{a}+{b}" for a, b in merges[:GATE["bpe_first_merges_checked"]]]
      == REC["text"]["bpe"]["first_merges"]
      and len(words) == GATE["bpe_word_level_vocabulary"]
      and all(w not in words and len(tx.apply_bpe(w, merges)) >= GATE["bpe_min_pieces_for_an_unseen_word"]
              for w in probes))
gate("A2", ok, f"the first {GATE['bpe_first_merges_checked']} merges reproduce the recorded run; "
               f"both probes are outside the {len(words):,}-word vocabulary and segment into pieces")

## A4. TF-IDF, first on eight sentences a reader can check by hand

$\mathrm{tf}_{d,w}$ is the share of document $d$'s tokens that are $w$. $\mathrm{idf}_w = \log(N / \mathrm{df}_w)$
where $\mathrm{df}_w$ counts the **fitting** documents containing $w$. Their product is large when a word is
frequent *here* and rare *elsewhere* — which is what "what this meeting said that others did not" means.

The same two formulas run over 245 statements below. First they run over eight sentences, so the arithmetic
is visible rather than asserted.

In [ ]:
toy = tx.toy_tfidf()
for name, sentence in toy["sentences"].items():
    print(f"{name}   {sentence}")
w = toy["weights"]
print()
print("document frequency and IDF, the four commonest terms and the four rarest:")
print(pd.concat([w.sort_values("documents", ascending=False).head(4),
                 w.sort_values("documents").head(4)]).to_string())
print("\nthe three heaviest terms of each sentence:")
print(toy["top_terms"].drop(columns="sentence").to_string())
print(f"\n'the' appears in {int(w.loc['the', 'documents'])} of 8 sentences, so its IDF is "
      f"{w.loc['the', 'IDF']:.3f} and it can never be anyone's top term.")
print(f"cosine(d1, d2) on raw counts is {toy['cosine_counts'].iloc[0, 1]:.3f}; on TF-IDF it is "
      f"{toy['cosine_tfidf'].iloc[0, 1]:.3f}.")
print("d1 cuts rates to support growth and d2 raises them to curb inflation. On counts they are close,")
print("because they share every function word. Weighting by rarity is what pulls them apart.")

Now the corpus. `fit_tfidf` is handed the 149 training statements and nothing else; `transform` is then
applied to all 245. The cost of not peeking is visible in the last two lines: about one token in ten of a
held-back statement falls outside the fitted vocabulary and is simply dropped.

In [ ]:
model = tx.fit_tfidf(train, min_df=GATE["tfidf_min_df"])     # vocabulary and IDF: the window only
X = model.transform(statements)                              # applied to all 245

print(f"vocabulary {len(model.vocabulary):,} terms (min_df = {model.min_df}, bare numerals dropped), "
      f"fitted on {model.n_fit} documents")
print(f"matrix {X.shape[0]} x {X.shape[1]}, {100 * (X == 0).mean():.1f}% of its entries zero")
print(f"token coverage: {100 * model.coverage(train):.1f}% inside the training window, "
      f"{100 * model.coverage(held_back):.1f}% on the held-back statements\n")

for date in ("2008-12-16", "2020-03-15", "2005-06-30"):
    row = [i for i, s in enumerate(statements) if s.date == date][0]
    print(f"{date}: " + ", ".join(tx.top_terms(X, model, row, k=8)))
print("\nThe heaviest terms are the news of the meeting, not its template text — which is the whole claim")
print("TF-IDF makes, and the reason the representation is worth more than a raw count.")

ok = (model.n_fit == GATE["tfidf_n_fit"] and len(model.vocabulary) == GATE["tfidf_vocabulary"]
      and max(s.date for s in train) <= GATE["corpus_training_end"]
      and X.shape == (GATE["corpus_statements"], GATE["tfidf_vocabulary"]))
gate("A3", ok, f"IDF computed from {model.n_fit} documents, all dated on or before "
               f"{GATE['corpus_training_end']}; {len(model.vocabulary):,} terms; applied to all "
               f"{X.shape[0]}")

The same matrix answers a question the slides ask of three toy documents: how alike are two real statements?
Each statement is compared with the one before it, under three weightings of the same fitted vocabulary:
raw counts, presence (0/1), and TF-IDF. The same pair of documents gets three different numbers.

In [ ]:
adj = tx.adjacent_similarity(statements, model)
rows = pd.DataFrame(adj["rows"])
print(f"{adj['pairs']} consecutive pairs; mean cosine on counts {adj['mean']['counts']:.2f}, "
      f"presence {adj['mean']['binary']:.2f}, TF-IDF {adj['mean']['tfidf']:.2f}")
print(f"pairs above 0.9: counts {100 * adj['share_above_090']['counts']:.0f}%, "
      f"TF-IDF {100 * adj['share_above_090']['tfidf']:.0f}%\n")
for label, r in (("December 2008", adj["december_2008"]), ("lowest TF-IDF", adj["lowest_tfidf"])):
    print(f"{label}: {r['previous']} -> {r['date']}   counts {r['counts']:.2f}   presence {r['binary']:.2f}   "
          f"TF-IDF {r['tfidf']:.2f}")
print("\nOn counts nearly every statement looks like the last one: the shared template text dominates the")
print("inner product. TF-IDF weighs the template text down, so what is left is what the meeting said that")
print("the previous one did not. Same documents, three answers: the weighting is part of the measure.")

The same rule, seen from the other side. If the IDF were fitted on all 245 statements, a word the Committee
used constantly after 2014 would weigh less in every earlier row: those rows would depend on statements not
yet written. The table puts the two fits side by side.

In [ ]:
look = pd.DataFrame(tx.idf_lookahead(statements, tx.LOOKAHEAD_TERMS)).set_index("term")
print(look[["df_window", "idf_window", "df_full", "idf_full"]].round(3).to_string())
b = look.loc["balance"]
print(f"\n'balance' in a 2010 row: IDF {b['idf_window']:.2f} fitted on the window, {b['idf_full']:.2f} fitted on")
print(f"everything — {100 * (1 - b['idf_full'] / b['idf_window']):.0f}% lower, because of statements written after 2014.")
print("'pandemic' has no column at all under the window fit: the word did not exist in it.")

## A5. Topics: latent Dirichlet allocation, by collapsed Gibbs sampling

A topic model compresses a document's $V$ counts into $K$ shares. Each topic $k$ is a distribution
$\phi_k$ over words; each statement has shares $\theta_i$ over topics; each token was written by first drawing
a topic from $\theta_i$ and then a word from that topic. Fitting runs this backwards. `fit_lda_gibbs` visits
every token, removes it from the counts, and redraws its topic with probability proportional to

$$\frac{n_k^{(v)}+b}{n_k^{(\cdot)}+Vb}\times\bigl(n_i^{(k)}+a\bigr),$$

how much topic $k$ likes word $v$, times how much statement $i$ already uses topic $k$. After 300 sweeps,
$\hat\phi$ and $\hat\theta$ are read off the counts. It sees the 149 training statements only. It takes about
a minute.

**The thermometer.** Every ten sweeps the sampler records the log-likelihood of the observed words under the
current bars and shares, $\sum_i \sum_{\text{tokens } v \text{ in } i} \log \sum_k \hat\theta_{ik}\hat\phi_{kv}$. It is a
monitor, not an objective: Gibbs maximises nothing. Random tags make every word improbable, so it starts low;
as the tags sort themselves it rises; once the chain has settled it is flat, and that is when the counts are read.

In [ ]:
t0 = time.time()
lda = tx.fit_lda_gibbs(train, K=GATE["lda_K"], sweeps=GATE["lda_sweeps"])
print(f"K = {lda.K}, a = {lda.a}, b = {lda.b}, {lda.sweeps} sweeps, seed {lda.seed}: "
      f"{len(lda.vocabulary)} words, {lda.tokens:,} tokens, {time.time() - t0:.0f} s")
for k in range(lda.K):
    print(f"topic {k + 1}: " + ", ".join(lda.top_words(k, 10)))

shares = lda.yearly_shares()
sweeps_at = [s for s in range(1, lda.sweeps + 1) if s % 10 == 0 or s == lda.sweeps]   # when the trace was taken
print(f"thermometer: {lda.loglik_trace[0]:,.0f} at sweep {sweeps_at[0]} -> {lda.loglik_trace[-1]:,.0f} at sweep {lda.sweeps}")
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [1, 1.7]})
ax0.plot(sweeps_at, lda.loglik_trace, color=TEAL)
ax0.set(xlabel="sweep", ylabel="log-likelihood of the words", title="the thermometer")
ax1.stackplot(shares.index.astype(int), shares.T.values, labels=shares.columns,
              colors=[TEAL, "#B38A38", RED, MUTED], alpha=0.6)
ax1.set_xlim(1994, 2014); ax1.set_ylim(0, 1); ax1.set_ylabel("mean topic share")
ax1.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), frameon=False)
plt.tight_layout(); plt.show()
print("Read the top words, then name each topic before looking at the plot. Then ask a neighbour to name")
print("them without seeing yours. Where you disagree, the name is a choice, not a finding.")
print("Then refit: tx.fit_lda_gibbs(train, K=4, seed=7) and tx.fit_lda_gibbs(train, K=6). Which topics")
print("survive the new seed, which split when K grows, and do the topic numbers stay put?")

ok = (lda.top_words(0, 8) == REC["text"]["lda"]["top_words"][0]
      and [lda.top_words(k, 8) for k in range(lda.K)] == REC["text"]["lda"]["top_words"]
      and len(lda.vocabulary) == GATE["lda_vocabulary"] and lda.tokens == GATE["lda_tokens"]
      and max(lda.dates) <= GATE["corpus_training_end"])
gate("A4", ok, f"{lda.K} topics fitted on {len(lda.dates)} statements, all dated on or before "
               f"{GATE['corpus_training_end']}; the top words reproduce the recorded run")

# Block B — Vectors, and a measurement that fails

A TF-IDF column knows nothing about any other column: "raise" and "increase" are as unrelated as "raise" and
"unemployment". **Word vectors** fix that by reading a word's company.

## B1. Word vectors: co-occurrence, PPMI, SVD

Three steps, all on the training window:

1. **Co-occurrence.** Count how often each pair of words appears within ±5 words of one another.
2. **PPMI.** Replace the count by $\max\{0, \log \frac{p(i,j)}{p(i)p(j)}\}$ — how much more often the pair
   occurs than chance would give. Zero for pairs no more frequent than chance, which is most of them.
3. **SVD.** Keep the first 100 singular directions, then normalise every row to unit length, so a dot product
   is a cosine and nothing depends on how often a word happens to appear.

`np.linalg.svd` on a dense matrix: deterministic, no seed, no randomised solver.

In [ ]:
emb = tx.fit_embeddings(train, window=GATE["embedding_window"], min_count=GATE["embedding_min_count"],
                        dim=GATE["embedding_dim"])
print(f"{len(emb.vocabulary)} words survive min_count = {emb.min_count} in {emb.n_fit} statements; "
      f"vectors are {emb.dim}-dimensional")
print(f"PPMI matrix is {100 * emb.ppmi_density:.1f}% non-zero; first singular values "
      f"{np.round(emb.singular_values[:6], 1)}\n")

for word in ("inflation", "employment", "committee", "accommodation"):
    print(f"{word:<15} " + ", ".join(f"{n} {v:.2f}" for n, v in tx.neighbors(emb, word, k=6)))

print("\nan analogy the vocabulary can carry, and one it cannot:")
print("   increase : increased :: moderate : " +
      ", ".join(f"{w} {v:.2f}" for w, v in tx.analogy(emb, "increase", "increased", "moderate", k=3)))
try:
    tx.analogy(emb, "raise", "raised", "lower")
except KeyError as exc:
    print(f"   raise : raised :: lower : ?  —  {exc}")
print("A word below min_count has no vector at all. That is a property of the method, not a defect of")
print("this run, and it is why the analogy table in the recorded run carries nulls rather than guesses.")

ok = (emb.n_fit == GATE["embedding_n_fit"] and len(emb.vocabulary) == GATE["embedding_vocabulary"]
      and emb.min_count == GATE["embedding_min_count"] and emb.dim == GATE["embedding_dim"])
gate("B1", ok, f"co-occurrence, PPMI and SVD from {emb.n_fit} training statements only; "
               f"{len(emb.vocabulary)} words at min_count {emb.min_count}")

## B2. Two measurements built on those vectors

**A hawk–dove axis.** Average the vectors of the hawkish seed words (raise, increase, increased, increasing,
tightening, firming), average the dovish ones (lower, easing, eased, accommodative, accommodation), subtract, and
normalise. Seeds seen fewer than ten times have no vector and drop out: five hawkish and four dovish remain. A statement is scored by the cosine between that axis and its TF-IDF-weighted mean word vector, so
template text carries little weight. Positive is meant to read hawkish.

**A novelty index.** $1 - \cos(x_t, \bar x_{t-8:t-1})$: how far this meeting's language sits from what the
Committee has been saying lately. It is not a stance proxy and is not offered as one; it answers a different
question, and it is here so that the two can be told apart.

In [ ]:
hawkdove = tx.signal_embedding_hawkdove(statements, emb, model)
dates = [s.date for s in statements]
print("hawk seeds inside the vocabulary:", {w: hawkdove["hawk_counts"][w] for w in hawkdove["hawk_used"]})
print("dove seeds inside the vocabulary:", {w: hawkdove["dove_counts"][w] for w in hawkdove["dove_used"]})
print("(counts are occurrences in the training window; 'raised' and 'firming' behave very differently"
      " from 'accommodation')\n")
print("the eight most novel statements:", ", ".join(f"{d} {v:.2f}" for d, v in hawkdove["most_novel"][:8]))

x = np.arange(len(dates))
fig, ax = plt.subplots(2, 1, figsize=(10, 4.4), sharex=True)
labels = tx.cycle_labels(dates)
for a in ax:
    for i in range(len(dates) - 1):
        if labels[i] != 0:
            a.axvspan(i, i + 1, color=TEAL if labels[i] > 0 else RED, alpha=0.12, lw=0)
ax[0].plot(x, hawkdove["score"], color=MUTED, lw=0.9)
ax[0].axhline(0, color="0.7", lw=0.6)
ax[0].set(ylabel="hawk–dove axis", title="shaded: hiking cycles (teal) and cutting cycles (red)")
ax[1].plot(x, hawkdove["novelty"], color=MUTED, lw=0.9)
ax[1].set(ylabel="novelty index", xlabel="meeting")
ticks = [i for i in range(0, len(dates), 30)]
ax[1].set_xticks(ticks); ax[1].set_xticklabels([dates[i][:7] for i in ticks])
plt.tight_layout(); plt.show()

## B3. Validation: against what the Committee then did

A stance read at meeting $t$ is a claim about where policy goes next, so the thing to line it up with is not
the label of meeting $t$ but the **subsequent policy path**: the mean cycle label over the next three meetings,
$+1$ inside a hiking band, $-1$ inside a cutting band, $0$ between cycles. The cycles are the Fed's own, dated
as Lab 7A dated them.

Two verdicts are reported for each signal. The **sign test** — every hiking band positive and every cutting
band negative — because the units of these signals are not comparable and a threshold would be arbitrary. And
**Spearman's rank correlation with the subsequent path**, which says how much of the ordering survives.

In [ ]:
regex = tx.signal_decision_sentence(statements, scope="statement")
checks = [tx.validate_signal(dates, hawkdove["score"], "embedding hawk-dove",
                             horizon=GATE["validation_horizon"]),
          tx.validate_signal(dates, regex["score"], "decision-sentence regex",
                             horizon=GATE["validation_horizon"])]
print(f"scored inside the training window only (end = {checks[0]['window_end']}), "
      f"{checks[0]['n_scored']} statements each\n")
print(f"{'signal':<26}{'hiking mean':>13}{'cutting mean':>14}{'gap':>9}"
      f"{'rank corr. w/ path':>21}{'verdict':>10}")
for v in checks:
    print(f"{v['signal']:<26}{v['hiking_mean']:>13.3f}{v['cutting_mean']:>14.3f}{v['gap']:>9.3f}"
          f"{v['rank_correlation_with_subsequent_path']:>21.3f}"
          f"{'PASSES' if v['passes'] else 'FAILS':>10}")
print("\nper cycle — where each signal works and where it does not:")
cycles = (tx.validation_table(checks).pivot(index="cycle", columns="signal", values="mean")
          .dropna(how="all")[[v["signal"] for v in checks]])
print(cycles.to_string())
print("The last four cycles begin after 2014 and sit outside the training window, so neither signal")
print("is scored on them here.")

tol = GATE["validation_tolerance"]
recorded = [REC["text"]["validation"][f"{v['signal']} | training window"] for v in checks]
ok = all(abs(v["rank_correlation_with_subsequent_path"]
             - r["rank_correlation_with_subsequent_path"]) < tol
         and v["passes"] == r["passes"] and v["n_scored"] == r["n_scored"] for v, r in zip(checks, recorded))
gate("B2", ok, f"both signals reproduce the recorded run to {tol:g}: rank correlations "
               f"{checks[0]['rank_correlation_with_subsequent_path']:+.3f} and "
               f"{checks[1]['rank_correlation_with_subsequent_path']:+.3f}")

In [ ]:
# Deck 5.2 centres document vectors before taking a cosine. Does centring rescue this score? No.
centred = tx.embedding_centred_check(train, emb, model)
print(f"mean cosine between two statements' document vectors: "
      f"{centred['mean_pairwise_cosine_of_document_vectors']:.2f}  (a large common part)")
for row in centred["rows"]:
    print(f"  {row['variant']:<32} rank correlation with the policy path {row['rank_correlation']:+.3f}   "
          f"{'passes' if row['passes'] else 'fails'}")
print("recorded:", [(r["variant"], r["rank_correlation"]) for r in REC["text"]["embedding_centred"]["rows"]])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.1))
for a, v in zip(ax, checks):
    rows = [r for r in v["per_cycle"] if r["covered"]]
    colour = [TEAL if r["kind"] == "hiking" else RED for r in rows]
    a.bar(range(len(rows)), [r["mean"] for r in rows], color=colour)
    a.axhline(0, color="k", lw=0.7)
    a.set_xticks(range(len(rows)))
    a.set_xticklabels([r["start"] for r in rows], rotation=45, ha="right", fontsize=7)
    a.set_title(f"{v['signal']}\n{'passes' if v['passes'] else 'FAILS'} the sign test", fontsize=9)
ax[0].set_ylabel("band mean")
plt.tight_layout(); plt.show()
print("A hiking band (teal) should stand above zero and a cutting band (red) below it.")
worst = checks[0]["worst_hiking_band"]
print(f"For the hawk–dove axis, the {worst['start']} to {worst['end']} hiking cycle — {worst['n']} meetings"
      f" — has mean {worst['mean']:+.4f}:")
print("the most negative band anywhere in the training window belongs to a two-year run of rate increases.")

## B4. This is the result, not a defect

The embedding hawk–dove score reads the 2004–06 tightening cycle as the most dovish stretch in the sample.
Nothing is broken. The axis measures **vocabulary**, and in 2004–06 the Committee's vocabulary was dovish
while its decision was not: it was *removing accommodation*, at a *measured* pace, and said so at every
meeting. A mean document vector built from the whole statement is pulled toward "accommodation" — a dove seed
word — precisely because the Committee was taking accommodation away.

The cell below counts the words. Then it prints the sentence the regex reads.

In [ ]:
band = [i for i, d in enumerate(dates) if "2004-06" <= d[:7] < "2006-07"]
rest = [i for i, d in enumerate(dates) if i not in band and d <= tx.TRAIN_END]
def per_statement(idx, group):
    return np.mean([sum(tx.BASELINE.run(statements[i].text).count(w) for w in group) for i in idx])

print(f"{'words':<46}{'2004-06 cycle':>15}{'rest of window':>17}")
for group in (["accommodation"], ["accommodative"], ["measured"], ["raise", "raised", "raising"],
              ["increase", "increased", "increasing"]):
    print(f"{', '.join(group):<46}{per_statement(band, group):>15.2f}{per_statement(rest, group):>17.2f}")
print("\nper statement. 'accommodation' nearly doubles; 'measured' goes from rare to almost every meeting;")
print("and the word that actually names the decision, 'raise', goes from 0.08 to 1.00 — once per statement.\n")

for i in (band[0], band[len(band) // 2], band[-1]):
    print(f"{dates[i]}  hawk {regex['hawk_matches'][i]:.0f} dove {regex['dove_matches'][i]:.0f}  |  "
          f"{tx.decision_sentence(statements[i].text)}")
print("\nThe regex is ten lines: a decision verb within six words of the policy instrument. It passes because")
print("it reads the one sentence that announces the decision, and ignores the paragraphs that explain it.")
print("Identification precedes measurement: deciding what sentence carries the quantity comes first, and no")
print("amount of representation learning substitutes for it.")

# Block C — Inside the small transformer

## C1. The arithmetic, by hand

Attention is a weighted average whose weights are computed from the data. For a query at position $i$:
score every position, $s_{ij} = \langle q_i, k_j\rangle / \sqrt{d_h}$; soften the scores into weights,
$a_{ij} = e^{s_{ij}} / \sum_k e^{s_{ik}}$; average the values with those weights. That is the whole mechanism.

Chapter 13's example box stipulates nine scores for the sentence *"The central bank raised interest rates
amid rising inflation"*, with "interest" as the query. **The chapter says in terms that the scores are set
for exposition and are not the output of any model**, so what is reproduced here is the arithmetic, not the
scores.

In [ ]:
book = at.book_attention()
print(f"query: position {book['query_position']}, '{book['query_token']}'  —  "
      f"{len(book['tokens'])} tokens\n")
print(f"{'position':<10}{'token':<12}{'score':>8}{'exp(score)':>13}{'weight':>10}")
for i, t in enumerate(book["tokens"], start=1):
    print(f"{i:<10}{t:<12}{book['scores'][t]:>8.1f}{book['exponentials'][t]:>13.4f}"
          f"{book['weights'][t]:>10.4f}")
print(f"{'':<22}{'sum':>8}{book['sum_of_exponentials']:>13.4f}{book['weights_sum']:>10.4f}")
print(f"\nentropy of the row: {book['entropy']:.4f} nats, against ln {len(book['tokens'])} = "
      f"{np.log(len(book['tokens'])):.4f} for a flat row\n")
print(f"{'quantity':<28}{'chapter prints':>16}{'computed here':>16}   agrees")
for row in book["comparison"]:
    print(f"{row['quantity']:<28}{row['printed']:>16}{row['computed_to_printed_precision']:>16}   "
          f"{'yes' if row['agrees'] else 'NO'}")
print("\n" + book["rounding_note"])
print("\n" + book["stipulation"])

ok = (book["all_agree"] and abs(book["sum_of_exponentials"] - GATE["book_attention_sum_of_exponentials"]) < 5e-4
      and len(book["comparison"]) == GATE["book_attention_digits_checked"])
gate("C1", ok, f"all {len(book['comparison'])} printed figures reproduce at the chapter's own precision; "
               f"the exponentials sum to {book['sum_of_exponentials']:.4f}")

In [ ]:
five = at.lab7b_five_score_variant()
nine = {row["quantity"]: row["computed_to_printed_precision"] for row in book["comparison"]}
print(five["note"])
print("   five-token softmax :", {k: f"{v:.2f}" for k, v in five["weights"].items()})
print("   nine-token softmax :", {k: nine[k] for k in five["weights"]})
print("  ", five["difference_from_the_nine_token_softmax"])
print("\nThe two are the same formula over different sets. A slide that shows both has to say which is which,")
print("because the four zero-scored positions carry 0.04 each and their absence lifts every other weight.")

## C1b. The same arithmetic with untrained weights, then masked

C1 started from stipulated scores. Here the scores are *computed*: give each of the nine tokens a random
16-number embedding, draw random $\mathbf W_Q, \mathbf W_K, \mathbf W_V$ (head width 8) and run the formula.
Nothing has been trained, so nothing tells *interest* which neighbours matter: its row comes out nearly flat,
close to $1/9$. The pattern in C1 is what training has to supply.

Then apply the **causal mask** a text generator needs: set $s_{ij} = -\infty$ for every $j > i$, so position $i$
cannot read the tokens after it. Those weights become exactly 0 and each row renormalises over $j \le i$.

In [ ]:
rng = np.random.default_rng(at.SEED)
X = rng.normal(size=(len(at.BOOK_TOKENS), 16)) / 4.0            # nine random embeddings, d = 16
W_Q, W_K, W_V, W_O = at.random_projections(16, 8, seed=at.SEED)  # untrained, head width d_h = 8
free = at.attention_from_embeddings(X, W_Q, W_K, W_V, W_O, causal=False)
masked = at.attention_from_embeddings(X, W_Q, W_K, W_V, W_O, causal=True)
q = at.BOOK_QUERY_POSITION - 1
book_masked = np.exp([at.BOOK_SCORES[t] for t in at.BOOK_TOKENS[:q + 1]])
book_masked = book_masked / book_masked.sum()

print(f"query '{at.BOOK_TOKENS[q]}' (position {q + 1})\n")
print(f"{'token':<12}{'untrained':>11}{'untrained, masked':>19}{'C1, masked':>12}")
for j, t in enumerate(at.BOOK_TOKENS):
    bm = f"{book_masked[j]:.3f}" if j <= q else "0"
    print(f"{t:<12}{free.weights[q, j]:>11.3f}{masked.weights[q, j]:>19.3f}{bm:>12}")
print(f"\nuntrained row: {free.weights[q].min():.3f} to {free.weights[q].max():.3f}, against 1/9 = {1/9:.3f}")
print(f"largest weight above the diagonal, masked: {np.triu(masked.weights, 1).max():.1e}")

rec = REC["attention"]["attention_by_hand"]
ok = (np.array_equal(np.round(free.weights[q], 4), np.array(rec["bidirectional"]["weights"]))
      and np.array_equal(np.round(masked.weights[q], 4), np.array(rec["causal"]["weights"]))
      and np.triu(masked.weights, 1).max() == 0.0)
gate("C1b", ok, "the untrained row and its masked version reproduce the recorded run at its four decimals; "
                "every weight above the diagonal is exactly 0")

## C2. Machine B, opened

Session 1 trained a character-level MiniGPT on these same 245 statements: $d_{\text{model}} = 128$, 4 heads,
3 layers, context 128 characters, a 78-character vocabulary, **631,424 parameters**. It is mirrored into
`session05_assets/minigpt/` under the course freeze manifest, and three things are proved before a single
weight is used: the five files match their recorded hashes; the class defined in `session05_attention.py`
produces exactly the state-dictionary keys and shapes the checkpoint carries; and the parameter count is the
one Session 1 reported. A checkpoint that loads with missing or unexpected keys is a different model wearing
the same filename.

One line was added to Session 1's model class and nothing else: `return_attn`, which hands back the attention
weights that every forward pass already computes and that Session 1 dropped on the floor.

In [ ]:
hashes = at.check_machine_b()
for name, digest in hashes.items():
    print(f"   {name:<24} {digest[:16]}…")
machine = at.load_machine_b()
print(f"\n{machine.parameters:,} parameters, config {machine.config}, vocabulary "
      f"{len(machine.vocabulary)} characters")
print(f"{len(machine.model.state_dict())} tensors in the state dictionary\n")
print(f"{'part':<34}{'parameters':>12}{'share':>8}")
for row in at.parameter_table(machine.model):
    print(f"{row['part']:<34}{row['parameters']:>12,}{100 * row['share']:>7.1f}%")

agree = at.corpus_agrees_with_training_report()
print(f"\ntraining corpus here: {agree['characters']:,} characters, sha256 {agree['sha256'][:16]}… — "
      f"{'the text Session 1 trained on' if agree['agrees'] else 'NOT the text Session 1 trained on'}")
report = machine.report
print(f"Session 1 ran {report['steps']:,} steps in {report['elapsed_seconds'] / 60:.0f} minutes and reached "
      f"a held-out loss of {report['final_validation_loss']:.3f}")

ok = (machine.parameters == GATE["machine_b_parameters"]
      and machine.config["context"] == GATE["machine_b_context"]
      and len(machine.vocabulary) == GATE["machine_b_vocabulary"]
      and machine.config["n_layers"] == GATE["machine_b_layers"]
      and machine.config["n_heads"] == GATE["machine_b_heads"]
      and len(machine.model.state_dict()) == GATE["machine_b_state_dict_keys"]
      and len(hashes) == GATE["machine_b_files"] and agree["agrees"])
gate("C2", ok, f"{len(hashes)} files match the freeze manifest; {machine.parameters:,} parameters, "
               f"{len(machine.model.state_dict())} state-dict tensors, corpus hash agrees")

## C3. What the heads look at

The rows of an attention map are query positions, the columns the positions they read. Above the diagonal the
weights are **exactly zero**: the model is causal and no position may read its own future. Each row's Shannon
entropy, in nats, says what the head is doing — $0$ is a head reading one position, $\ln T$ a head averaging
over everything it can see.

In [ ]:
amap = at.attention_map(machine, at.DEMO_SENTENCE)
print(f'"{at.DEMO_SENTENCE}"  —  {amap["characters"]} characters, '
      f'{amap["n_layers"]} layers x {amap["n_heads"]} heads')
print(f"flat would be ln {amap['characters']} = {amap['max_possible_entropy']:.2f} nats\n")
print(f"{'mean row entropy':<20}" + "".join(f"{'head ' + str(h + 1):>10}" for h in range(amap["n_heads"])))
for l, row in enumerate(amap["entropy"], start=1):
    print(f"{'layer ' + str(l):<20}" + "".join(f"{v:>10.2f}" for v in row))
print(f"\nlargest weight above the diagonal: {amap['largest_weight_above_the_diagonal']:.1e}")
print("Layer 1 stays diffuse across all four heads; by layer 2 at least one head has collapsed onto a few")
print("positions. Specialisation is learned, and it is learned in that order.")

low_l = int(np.argmin([min(r) for r in amap["entropy"]]))
low_h = int(np.argmin(amap["entropy"][low_l]))
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
for a, (l, h, title) in zip(ax, [(0, 0, "layer 1, head 1"),
                                 (low_l, low_h, f"layer {low_l + 1}, head {low_h + 1} (lowest entropy)")]):
    im = a.imshow(amap["maps"][l][h], cmap="magma", vmin=0, vmax=1)
    a.set(title=title, xlabel="position read", ylabel="query position")
    fig.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout(); plt.show()

rows = np.concatenate([m.reshape(-1, m.shape[-1]) for m in amap["maps"]])
row_error = float(np.abs(rows.sum(axis=1) - 1.0).max())
ok = (amap["largest_weight_above_the_diagonal"] <= GATE["attention_weight_above_diagonal_max"]
      and row_error < GATE["attention_row_sum_error_max"]
      and amap["n_layers"] == GATE["machine_b_layers"] and amap["n_heads"] == GATE["machine_b_heads"])
gate("C3", ok, f"every weight above the diagonal is {amap['largest_weight_above_the_diagonal']:.0e}; "
               f"every row sums to 1 within {row_error:.1e}")

## C4. What it writes, and the one dial

Generation is: predict the next character, divide the logits by the **temperature**, draw, append, repeat.
Below one, the distribution sharpens and the model repeats the house phrasing. At one it is the distribution
the model actually learned. Above one the tail gets its share and the spelling goes.

In [ ]:
for temperature in at.TEMPERATURES:
    text = at.generate(machine, prompt="The Committee decided", n_new=200, temperature=temperature)
    print(f"temperature {temperature}")
    print("   " + text.replace("\n", " / "))
    print()
print("One character at a time from a 0.6M-parameter model. The spelling, the house phrasing and the shape of")
print(f"a policy sentence are all learned from {machine.report['corpus_characters']:,} characters of FOMC "
      f"prose;")
print("the arithmetic of a rate decision is not, and nothing in the training objective asked for it.")

## C5. Sixty steps from noise

The checkpoint above is a result. This is the process. A **freshly initialised** model of the same
architecture is trained on the same corpus for sixty steps with Session 1's own recipe — AdamW, lr $3\times10^{-4}$,
linear warm-up over 100 steps, gradients clipped at 1.0, batches of 32 windows of 128 characters.

Sixty steps is not a small version of Session 1's three thousand; it is the first inch of them. The point is
the starting value: a model that knows nothing picks uniformly among 78 characters and pays
$\ln 78 = 4.357$ nats per character. Watch it leave that line.

The held-out split is by **whole statement** — every twentieth one — never by random window, so a validation
loss cannot be read off a sentence the model already met in training.

In [ ]:
t0 = time.time()
fresh = at.train_a_few_steps(machine, n_steps=60)
print(f"{fresh['n_steps']} steps in {time.time() - t0:.0f} s on CPU, {fresh['parameters']:,} parameters "
      f"from a fresh initialisation")
print(f"loss: uniform {fresh['uniform_loss']:.3f}  ->  step 1 {fresh['first_loss']:.3f}  ->  "
      f"step {fresh['n_steps']} {fresh['last_loss']:.3f}   (held out: "
      f"{fresh['first_validation']:.3f} -> {fresh['last_validation']:.3f})")
d = fresh["data"]
print(f"split: {d['n_held_out']} of {d['n_statements']} statements held out, "
      f"{d['train_characters']:,} training characters and {d['validation_characters']:,} held-out")

steps, losses = zip(*fresh["train_curve"])
vsteps, vlosses = zip(*fresh["validation_curve"])
plt.figure(figsize=(6, 3.2))
plt.plot(steps, losses, color=MUTED, lw=0.9, label="training batch")
plt.plot(vsteps, vlosses, "o-", color=TEAL, ms=3, label="held-out statements")
plt.axhline(fresh["uniform_loss"], color=RED, ls="--", lw=0.9,
            label=f"uniform over 78 characters, ln 78 = {fresh['uniform_loss']:.2f}")
plt.axhline(machine.report["final_validation_loss"], color="0.6", ls=":", lw=0.9,
            label=f"Session 1 after {machine.report['steps']:,} steps")
plt.xlabel("step"); plt.ylabel("cross-entropy, nats per character"); plt.legend(frameon=False, fontsize=7)
plt.tight_layout(); plt.show()
print("after 60 steps, the fresh model writes:")
print("   " + fresh["sample_after"].replace("\n", " / "))
print("It has learned the letter frequencies and the space. It has not learned a word.")

# Every character of every statement has to land in exactly one of the two streams. Each stream is
# the statements joined by a blank line, so subtract the separators before comparing the totals.
n_train = d["n_statements"] - d["n_held_out"]
bodies = d["characters"] - 2 * (d["n_statements"] - 1)
split_bodies = ((d["train_characters"] - 2 * (n_train - 1))
                + (d["validation_characters"] - 2 * (d["n_held_out"] - 1)))
accounting = bodies - split_bodies
print(f"characters in the {d['n_statements']} statement bodies: {bodies:,}; in the two streams: "
      f"{split_bodies:,}; unaccounted for: {accounting}")
ok = (fresh["parameters"] == GATE["fresh_model_parameters"]
      and d["n_held_out"] == GATE["fresh_model_held_out_statements"]
      and abs(accounting) <= GATE["fresh_model_character_accounting_error_max"])
gate("C4", ok, f"the fresh model is the same architecture ({fresh['parameters']:,} parameters); "
               f"{d['n_held_out']} whole statements held out and every character accounted for")

# Block D — The recorded language-model instrument

A language model asked to label a statement is an **instrument**. Like any instrument it has a reading, and
like any instrument the reading is worthless without provenance and without a measure of how much it moves
when nothing it is measuring has moved.

Four prompt variants were recorded outside this notebook, by a separate instructor process, on 2026-09-20:

| variant | what it is | statements |
|---|---|---|
| **A** | the frozen prompt | all 245 |
| **B** | the same task, **reworded** | 48 |
| **C** | the same words, **options reordered** (DOVISH first) | 48 |
| **D** | **byte-identical to A**, run again | 48 |

A against B and C measures prompt sensitivity. A against D measures nothing but **sampling noise**: the input
was the same bytes. This notebook reads the record; it never calls a model and never touches the network.

In [ ]:
STANCE = ASSETS / "llm_stance"
lines = (STANCE / "raw_replies.jsonl").read_text(encoding="utf-8").splitlines()
replies = pd.DataFrame([json.loads(line) for line in lines if line.strip()])
replies["model"] = replies["model_used"].map(lambda v: v[0] if isinstance(v, list) else v)

print("provenance of every score in this block")
print(f"   model returned   : {sorted(replies['model'].unique())}")
print(f"   harness          : {sorted(replies['harness'].unique())}")
print(f"   recorded         : {replies['recorded_at'].min()[:10]}, "
      f"{replies['recorded_at'].min()[11:]} to {replies['recorded_at'].max()[11:]}")
print(f"   replies          : {len(replies)}   unparsed or errored: {int(replies['error'].notna().sum())}"
      f"   cost: ${replies['cost_usd'].sum():.2f}")
print("   prompt sha256-16 : " +
      "   ".join(f"{v} {h}" for v, h in replies.groupby("variant")["prompt_sha256_16"].first().items()))
print("                      A and D carry one hash because D is A's bytes, run a second time.\n")

print("the first eight variant-A scores, each beside what produced it:")
print(f"{'date':<12}{'label':<10}{'model':<18}{'harness':<24}{'recorded':<12}prompt")
for _, r in replies[replies["variant"] == "A"].sort_values("date").head(8).iterrows():
    print(f"{r['date']:<12}{r['label']:<10}{r['model']:<18}{r['harness']:<24}"
          f"{r['recorded_at'][:10]:<12}{r['prompt_sha256_16']}")
print("\nNo number a model produced appears anywhere in Session 5 without those four fields beside it.")

by_variant = replies.groupby("variant")["prompt_sha256_16"].first()
a, dvar = GATE["llm_identical_prompt_pair"]
ok = (len(replies) == GATE["llm_replies"]
      and replies["variant"].nunique() == GATE["llm_variants"]
      and int(replies["error"].notna().sum()) <= GATE["llm_errors_max"]
      and all(replies[f].notna().all() for f in GATE["llm_reply_fields"])
      and by_variant[a] == by_variant[dvar]
      and len({by_variant[a], by_variant["B"], by_variant["C"]}) == 3)
gate("D1", ok, f"{len(replies)} replies, none errored, each carrying "
               f"{', '.join(GATE['llm_reply_fields'])}; {a} and {dvar} share one prompt hash and "
               f"B and C each differ")

## D2. Agreement, and Cohen's κ

Raw agreement flatters an instrument that mostly says the same thing. **Cohen's κ** subtracts the agreement
two raters would reach by chance given their own label frequencies: $\kappa = (p_o - p_e)/(1 - p_e)$.

κ is computed here by `cohen_kappa` from `book_code/ch14/human_llm_kappa.py` — the chapter's own function,
hash-checked against `book_code/MANIFEST.json` before it is allowed to run.

In [ ]:
scores = pd.read_csv(STANCE / "scores.csv")
wide = scores.pivot(index="date", columns="variant", values="label")
common = wide.dropna()
hk = me.book_module("human_llm_kappa")             # the chapter's own kappa, hash-checked on import

what = {"AB": "reworded prompt", "AC": "options reordered", "AD": "byte-identical repeat",
        "BC": "reworded vs reordered", "BD": "reworded vs repeat", "CD": "reordered vs repeat"}
rows = []
for p, q in itertools.combinations(sorted(wide.columns), 2):
    u, v = common[p].to_numpy(), common[q].to_numpy()
    rows.append(dict(pair=f"{p} vs {q}", what=what[p + q], agreement=float((u == v).mean()),
                     kappa=float(hk.cohen_kappa(u, v))))
pairs = pd.DataFrame(rows)
print(f"all four variants cover the same {len(common)} statements\n")
print(f"{'pair':<10}{'what it varies':<26}{'agreement':>11}{'kappa':>9}   {'Fleiss band'}")
for r in rows:
    print(f"{r['pair']:<10}{r['what']:<26}{100 * r['agreement']:>10.1f}%{r['kappa']:>9.3f}   "
          f"{hk.fleiss_band(r['kappa'])}")

ad = pairs.set_index("pair").loc["A vs D"]
ab = pairs.set_index("pair").loc["A vs B"]
print(f"\nThe byte-identical repeat, A vs D, agrees LESS than the reworded prompt, A vs B: "
      f"kappa {ad['kappa']:.3f} against {ab['kappa']:.3f}.")
print("On this corpus, with this model, on this day, resampling the same prompt moves the label at least as")
print("much as rewording it. Sampling noise dominates wording here.")

ok = (len(common) == GATE["llm_common_dates"]
      and sorted(pd.unique(scores["label"])) == GATE["llm_labels"]
      and int((scores["variant"] == "A").sum()) == GATE["llm_variant_a_scored"]
      and len(wide.columns) == GATE["llm_variants"])
gate("D2", ok, f"the four variants are compared on the same {len(common)} statements, "
               f"labels confined to {GATE['llm_labels']}, variant A covering all "
               f"{GATE['llm_variant_a_scored']}")

In [ ]:
print("where A and D disagree (rows: A's label; columns: the same prompt, run again)")
table = pd.crosstab(common["A"], common["D"])
print(table.to_string())
moved = int(len(common) - np.trace(table.to_numpy()))
print(f"\n{moved} of {len(common)} statements changed label when nothing about the input changed.")
print("A researcher who ran this prompt once, and reported the labels, would be reporting one draw from")
print("that table. Neither column is more correct than the other; that is the point.")
print("\nRead the ordering carefully. It says sampling noise is large HERE — 48 statements, one model, one")
print("day, three coarse labels. It is not a general claim that wording does not matter, and a study with a")
print("bigger wording contrast or a finer label set could easily reverse it. What it does establish is that")
print("prompt-sensitivity checks that never repeat the same prompt cannot tell the two apart at all.")

## D3. Does the reading line up with what the Committee then did?

Same validation as block B: the score against the subsequent policy path, per cycle, inside the training
window so that it is comparable with the other signals.

In [ ]:
llm_signal = tx.signal_llm_stance(statements)          # reads scores.csv; variant A; +1 / 0 / -1
llm_check = tx.validate_signal(dates, llm_signal["score"], "language-model stance",
                               horizon=GATE["validation_horizon"])
print(f"{llm_signal['n_scored']} of {llm_signal['n_statements']} statements carry a variant-A score\n")
print(f"{'signal':<26}{'hiking mean':>13}{'cutting mean':>14}{'rank corr. w/ path':>21}{'verdict':>10}")
for v in (llm_check, checks[1], checks[0]):
    print(f"{v['signal']:<26}{v['hiking_mean']:>13.3f}{v['cutting_mean']:>14.3f}"
          f"{v['rank_correlation_with_subsequent_path']:>21.3f}"
          f"{'PASSES' if v['passes'] else 'FAILS':>10}")
print("\nper cycle, over the cycles the training window covers:")
per_cycle = tx.validation_table([llm_check]).drop(columns="signal").dropna(subset=["mean"])
print(per_cycle.to_string(index=False))

r = REC["text"]["validation"]["language-model stance | training window"]
ok = (abs(llm_check["rank_correlation_with_subsequent_path"]
          - r["rank_correlation_with_subsequent_path"]) < GATE["validation_tolerance"]
      and llm_check["passes"] == r["passes"] and llm_check["n_scored"] == r["n_scored"])
gate("D3", ok, f"the recorded instrument reproduces the recorded run to "
               f"{GATE['validation_tolerance']:g}: rank correlation "
               f"{llm_check['rank_correlation_with_subsequent_path']:+.3f} on "
               f"{llm_check['n_scored']} statements")

## D4. What has been established, and what has not

Two things were measured in this block, and neither of them is validity.

1. **Consistency.** How much the reading moves when the instrument is re-applied to the same object, and when
   the instrument is reworded. Both were large, and the repeat moved more than the rewording.
2. **Convergent validity.** Whether the reading lines up with an independent quantity it ought to line up
   with — here, the Fed's own subsequent policy path. It does, better than the embedding axis and less well
   than the regex on the decision sentence.

What is **not** here is validity against a gold standard. That would require a set of statements labelled by
people, under a written coding manual, with their disagreements recorded — and **no human gold labels exist in
this build**. Nobody hand-coded these 245 statements. Every κ in this block is model against model.

Without gold labels there is no way to say whether the instrument is *biased*, only whether it is *stable* and
whether it *agrees with something else*. An instrument can be perfectly stable and consistently wrong; block D
cannot tell those apart. Block E is what buying the missing piece costs, and what it buys.

# Block E — What validation buys

Suppose the reading goes into a regression as a regressor: $y_i = \alpha + \beta m^{*}_i + \gamma' c_i + u_i$,
with $\hat m_i = m^{*}_i + \varepsilon_i$ in place of $m^{*}_i$. Chapter 14 works this through, and the four
scripts of that chapter are vendored byte-identically under `book_code/ch14/`. The cells below **import those
scripts and call their own functions in their own order** — nothing here re-derives a statistic, and a hash
check stops the run if a vendored file has been edited.

Each block is gated against the digits printed in `chapters/chapter14.tex`, at the precision the chapter
prints them. A disagreement would be recorded and left standing; nothing in `session05_measure.py` is tuned
towards the book.

## E1. Four estimators of the same $\beta$

In [ ]:
bias = me.bias_correction_table()
print(f"truth: beta = {bias.beta_true:.2f} on n = {bias.n:,} documents, of which "
      f"{bias.n_validation} are hand-coded")
print(f"corr(m*, m-hat) = {bias.corr_mstar_mhat:.3f}; mean error {bias.mean_bias:+.4f} — close to zero")
print(f"but the error loads on a covariate: slope on size {bias.error_on_size_slope:+.4f} "
      f"(s.e. {bias.error_on_size_se:.4f})\n")
print(f"{'estimator':<20}{'beta':>9}{'s.e.':>9}{'95% interval':>22}{'width':>9}{'% of truth':>12}")
for name, row in bias.estimators.items():
    print(f"{name:<20}{row['beta']:>9.4f}{row['se']:>9.4f}"
          f"{'[' + format(row['ci_low'], '.3f') + ', ' + format(row['ci_high'], '.3f') + ']':>22}"
          f"{row['ci_width']:>9.4f}{row['pct_of_truth']:>11.0f}%")
c = bias.calibration
print(f"\ncalibration on the {bias.n_validation} hand-coded rows: "
      f"m* = {c['intercept']:.3f} + {c['slope_m_hat']:.3f} m-hat + {c['slope_c']:.3f} c")
print(f"the corrected interval is {bias.corrected_narrower_pct:.0f}% narrower than using the hand-coded "
      f"rows alone.")
print("\nRead the table downward. Naive is what a researcher gets by treating the model's score as the")
print("variable: attenuated, and the mean error being near zero did not save it. Validation-only is")
print("unbiased and imprecise. Corrected calibrates the score on the hand-coded rows and then uses it on")
print("the full sample, buying the precision back.")

fig, ax = plt.subplots(figsize=(6.4, 2.6))
for i, (name, row) in enumerate(bias.estimators.items()):
    ax.plot([row["ci_low"], row["ci_high"]], [i, i], color=MUTED, lw=2)
    ax.plot(row["beta"], i, "o", color=TEAL, ms=5)
ax.axvline(bias.beta_true, color=RED, ls="--", lw=1, label=f"truth, {bias.beta_true:.2f}")
ax.set_yticks(range(len(bias.estimators))); ax.set_yticklabels(list(bias.estimators))
ax.invert_yaxis(); ax.set_xlabel("beta"); ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

## E2. How many hand-coded documents is enough

The width of the corrected interval against the size of the hand-coded set. The curve falls, and then it
stops falling: past a few hundred documents the binding constraint stops being the calibration equation's own
error and becomes the full-sample residual variance. That is the budget rule.

In [ ]:
sweep = me.validation_size_sweep()
print(f"{'hand-coded n':>14}{'beta':>10}{'95% interval':>22}{'width':>9}{'calibration slope':>20}")
for row in sweep.rows:
    print(f"{row['n_validation']:>14}{row['beta_corrected']:>10.4f}"
          f"{'[' + format(row['ci_low'], '.3f') + ', ' + format(row['ci_high'], '.3f') + ']':>22}"
          f"{row['ci_width']:>9.4f}{row['calibration_slope_m_hat']:>20.4f}")
print(f"\n50 -> 200 labels narrows the interval by {sweep.narrowing_50_to_200_pct:.0f}%; "
      f"200 -> 800 by a further {sweep.narrowing_200_to_800_pct:.0f}%.")
print("Four times the hand-coding for a fifth of the gain: the second four hundred documents are the")
print("expensive ones, and a design calculation is how you find that out before paying for them.")

plt.figure(figsize=(5.4, 2.8))
plt.plot([r["n_validation"] for r in sweep.rows], [r["ci_width"] for r in sweep.rows],
         "o-", color=TEAL, ms=4)
plt.xscale("log"); plt.xlabel("hand-coded documents"); plt.ylabel("width of the 95% interval")
plt.tight_layout(); plt.show()

digits = bias.gates + sweep.gates
gate("E1", all(g.agrees for g in digits) and len(digits) == GATE["chapter14_digits_estimation"],
     f"{sum(g.agrees for g in digits)} of {len(digits)} digits printed in chapter 14 reproduce here")

## E3. An overall κ that hides the failure

Chapter 14's table 14.2: a model and a human coder on four narrative categories. Overall κ reads as excellent
on the Fleiss bands. One class — the rare one, which is where the interesting hypothesis usually lives — is
coded far worse, and the overall figure cannot show it. Merging that class upward raises the overall number
without improving a single coding decision: it removes the distinction the research question was about.

In [ ]:
pck = me.per_class_kappa()
print(f"{pck.n_audit} hand-coded rows out of {pck.n_full:,}; accuracy {100 * pck.accuracy:.0f}%, "
      f"overall kappa {pck.kappa_overall:.3f}")
print(f"which Fleiss calls '{pck.fleiss_band}' and Landis-Koch '{pck.landis_koch_band}'\n")
print(f"{'class':<18}{'share':>8}{'support':>9}{'recall':>9}{'kappa':>9}   above 0.75?")
for row in pck.classes:
    print(f"{row['label']:<18}{100 * row['share']:>7.1f}%{row['support']:>9}{row['recall']:>9.2f}"
          f"{row['kappa']:>9.3f}   {'yes' if row['meets_threshold_0_75'] else 'NO'}")
m = pck.merge
print(f"\nmerging '{m['from']}' into '{m['into']}': kappa {m['kappa_before']:.3f} -> "
      f"{m['kappa_after']:.3f}")
print(f"what it costs: {m['what_it_costs']}")

## E4. A screen that cannot see the errors you most fear

Self-consistency screening asks the model the same question several times and flags the claims whose answers
move. Read the last line of the output carefully: resampling measures **how much the answer moves**, not
whether it is true. A false statement the model holds firmly resamples as consistently as a true one.

In [ ]:
sc = me.self_consistency()
print(f"{sc.n_claims} claims, {sc.resamples} resamples each\n")
print(f"{'kind of claim':<22}{'n':>6}{'mean h':>10}{'s.d.':>8}{'p10':>8}{'p90':>8}")
for name, row in sc.by_kind.items():
    print(f"{name:<22}{row['n']:>6}{row['mean_h']:>10.3f}{row['sd_h']:>8.3f}{row['p10']:>8.3f}"
          f"{row['p90']:>8.3f}")
print(f"\nbest threshold {sc.threshold:.2f}: precision {sc.precision:.2f}, recall {sc.recall:.2f}, "
      f"F1 {sc.f1:.2f}, flagging {100 * sc.flagged_share:.0f}% of claims")
print(f"fabricated claims missed: {sc.fabricated_missed} of {sc.fabricated_total}")
print(f"CONFIDENTLY WRONG claims — held firmly, and false — that the screen never flags: "
      f"{sc.stable_error_undetected} of {sc.stable_error_total}, {sc.stable_error_undetected_pct:.1f}%")
print("\nRaising the number of resamples does not help. It sharpens the score; it does not change what the")
print("score measures. The screen is a triage tool, not a test of truth.")

## E5. Design: how many items before an order effect can be told from zero

Two questions, one calculation. Which presentation removes a first-position effect — and how many items are
needed before the effect can be distinguished from zero.

In [ ]:
oe = me.order_effect_power()
print(f"{'design':<16}{'mean bias':>12}{'RMSE':>10}")
for name, row in oe.designs.items():
    print(f"{name:<16}{row['mean_bias']:>12.4f}{row['rmse']:>10.4f}")
print("\nHolding the order fixed leaves the whole effect standing. Randomising the order across items removes")
print("it. Exhausting the orderings at a fixed budget buys variance, not unbiasedness.\n")
print(f"{'effect size':>12}   {'setting':<17}{'items for 80% power':>22}{'simulated power':>18}")
for row in oe.deltas:
    print(f"{row['delta']:>12.3f}   {(row['setting'] or '—'):<17}{row['n_required_80pct_power']:>22,}"
          f"{row['simulated_power_at_n']:>18.2f}")
print(f"\npower at 50 items against the letter-label effect: {oe.letter_power_at_50_items:.2f}")
print("The requirement swings by a factor of forty across two label formats. A robustness check run on a few")
print("dozen items is evidence of nothing, in either direction.")

digits = pck.gates + sc.gates + oe.gates
gate("E2", all(g.agrees for g in digits) and len(digits) == GATE["chapter14_digits_agreement"],
     f"{sum(g.agrees for g in digits)} of {len(digits)} digits printed in chapter 14 reproduce here "
     f"({GATE['chapter14_digits_total']} across the block)")

# Block F — Does the Committee's language improve the forecast?

Session 2 built a one-month-ahead forecast of the 10-year Treasury yield from numbers alone. Block F asks
whether the FOMC's own words, available at the moment the forecast is made, improve on it.

**The design was written out in full and hashed before the test window was scored once.**
`protocol_forecast.md` is that design; `protocol_forecast.sha256` carries the hash and the moment it was
taken. The hash exists so that a reader can tell the difference between a design and a rationalisation.
The protocol says, in section 10.2, that "text adds nothing" is an acceptable, expected and fully reportable
outcome, and that **no gate anywhere in block F depends on text helping**.

## F1. The inputs, and the bar to beat

In [ ]:
machine_b = fc.assert_no_machine_b()          # protocol section 8.2
protocol_hash = fc.file_sha256(fc.PROTOCOL)
taken = [ln for ln in fc.PROTOCOL_SHA.read_text(encoding="utf-8").splitlines() if ln.startswith("Taken")]
print(f"protocol_forecast.md  sha256 {protocol_hash[:32]}…")
print(f"   matches the hash recorded beside it: {protocol_hash == GATE['forecast_protocol_sha256']}")
print(f"   {taken[0] if taken else ''}")
print(f"   that hash predates the first scored run: "
      f"{REC['forecast']['_meta']['protocol_hash_predates_first_scored_run']}\n")

inputs = fc.load_inputs()                     # both files hash-asserted at load; read-only
for name, digest in inputs.hashes.items():
    print(f"   {name:<26} {digest[:32]}…  matches protocol: "
          f"{digest == GATE['forecast_input_sha256'][name]}")
print(f"\npanel {len(inputs.panel)} rows, frozen forecasts {len(inputs.baseline)} rows")
print(f"the panel has no row for {', '.join(inputs.missing_forecast_dates)} — the only month-end missing "
      f"anywhere in it.")
print("Block F inherits that gap and never fills it. An interpolated month would be a month Session 2's")
print("published baselines were never scored on, and the two would stop being comparable.\n")

frozen = fc.frozen_baseline_table(inputs.baseline)
print(f"{'Session 2, scored as shipped':<30}{'RMSE':>9}{'MAE':>9}   reproduces to four decimals")
for name in ("random_walk", "ar1", "ols"):
    s = frozen["scored"][name]
    print(f"{name:<30}{s['rmse']:>9.4f}{s['mae']:>9.4f}   "
          f"{'yes' if frozen['reproduces_to_four_decimals'][name] else 'NO'}")
mlp = frozen["mlp"]
print(f"{'MLP, as Session 2 publishes it':<30}{mlp['session02_published']['rmse']:>9.4f}"
      f"{mlp['session02_published']['mae']:>9.4f}   mean of ten per-seed RMSEs")
print(f"{'MLP, the averaged column':<30}{mlp['column_as_scored']['rmse_here']:>9.4f}"
      f"{mlp['column_as_scored']['mae_here']:>9.4f}   mean forecast across ten seeds")
print("\nTwo different objects, gated against neither: averaging ten forecasts beats the average of the ten")
print("models. Session 5 quotes Session 2's published definition as the bar. Note what that bar is — the")
print("random walk and the AR(1) beat OLS-5 on this window, so any text model has to improve on a very")
print("low-variance forecast, not on the weakest structured model available.")

ok = (all(inputs.hashes[n] == GATE["forecast_input_sha256"][n] for n in inputs.hashes)
      and protocol_hash == GATE["forecast_protocol_sha256"]
      and inputs.missing_forecast_dates == GATE["forecast_missing_forecast_dates"]
      and all(frozen["reproduces_to_four_decimals"].values()))
gate("F1", ok, "both Session 2 files match the protocol's hashes, the 2025-11-30 gap is present and "
               "unfilled, and the three reproducible baselines come back to four decimals")

## F2. The estimation sample, the information rule, and the fitting discipline

The panel starts in 1989; the first statement is dated 1994-02-04. Fifty of Session 2's 300 training rows
predate every statement and could carry no text signal under any rule. The protocol resolves that by
**restriction, not imputation**: the estimation sample is every row with `forecast_made >= 1994-02-28`, which
leaves 250 / 60 / 79. Because the comparator is no longer estimated on what Session 2 estimated it on, every
result is reported in **three columns** — frozen file, re-estimated, plus text — so that the cost of the
restriction is never read as a text effect.

**The information rule.** Month $t$'s signal is the last statement dated on or before $t$'s last observation
date. Seven months carry two or three statements and the last one wins. Five releases are hand-listed and
removed as not being policy statements at all. In a month with no meeting the stance is carried forward,
novelty is **zero** — there is no new statement to be novel — and months-since-statement enters as its own
regressor so that staleness is measured rather than smuggled.

In [ ]:
corpus = fc.load_statements()                 # hash-asserted file by file; exclusion set applied
print(f"{len(corpus.dates)} of {GATE['corpus_statements']} releases kept; "
      f"{len(corpus.excluded)} hand-listed and removed:")
for e in corpus.excluded:
    print(f"   {e['date']}  " + textwrap.fill(e["reason"], width=94, subsequent_indent=" " * 15))

frame = fc.merge_text(inputs.panel, corpus)                # information rule; asserts no look-ahead
space = fc.fit_text_space(corpus)                          # vocabulary, IDF, SVD, seed axis
frame = fc.attach_features(frame, corpus, space)
parts = fc.split(frame)
sizes = {k: len(v) for k, v in parts.items()}
print(f"\nsplits {sizes}")
print(f"text space: {space.vocabulary_size:,} terms from {space.n_training_documents} documents dated on "
      f"or before {space.training_cutoff}")
print(f"seed axis: {len(space.seeds_found['hawkish'])} hawkish and {len(space.seeds_found['dovish'])} "
      f"dovish terms found in that vocabulary")
print(f"stale rows: train {int(parts['train']['is_stale'].sum())}, valid "
      f"{int(parts['valid']['is_stale'].sum())}, test {int(parts['test']['is_stale'].sum())} of 79")
print(f"novelty in a stale month: {float(frame.loc[frame['is_stale'], 'novelty'].abs().max()):.1f}\n")

latest_fitted = max(d for d in corpus.dates if d <= pd.Timestamp(fc.TEXT_TRAIN_END))
print(f"latest document inside any fitted object: {latest_fitted.date()}  "
      f"(cutoff {fc.TEXT_TRAIN_END})")
print(f"latest statement reaching any row: {frame['statement_date'].max().date()}, and no row carries a "
      f"statement dated after its own forecast date: "
      f"{bool((frame['statement_date'] <= frame['date']).all())}")
print(f"Machine B: artefacts loaded = {machine_b['machine_b_artefacts_loaded']}, "
      f"Session 1 modules in this process = {machine_b['session01_modules_in_process']}")
print("Block C opened Machine B in this kernel, and that object is not among block F's declared inputs:")
print("   declared inputs: baseline_predictions.csv, yield_panel.csv, fomc_manifest.json, scores.csv")
print(f"   excluded by assertion: {', '.join(fc.MACHINE_B_ARTEFACTS)}")
print("Machine B was trained on all 245 statements, the test window included; anything derived from it would")
print("be contaminated end to end, and the contamination would be invisible in a forecast metric.")

ok = (sizes == GATE["forecast_split_sizes"]
      and len(corpus.excluded) == GATE["forecast_excluded_releases"]
      and space.training_cutoff == GATE["forecast_text_training_cutoff"]
      and latest_fitted <= pd.Timestamp(GATE["forecast_text_training_cutoff"])
      and bool((frame["statement_date"] <= frame["date"]).all())
      and float(frame.loc[frame["is_stale"], "novelty"].abs().max()) == 0.0
      and not machine_b["machine_b_artefacts_loaded"])
gate("F2", ok, f"splits {sizes}; nothing after {fc.TEXT_TRAIN_END} in a fitted object; no statement "
               f"dated after its own row; novelty zero in every stale month; no Machine B artefact")

## F3. Scored once

The validation metrics are recomputed live — they are cheap and carry no inferential weight. The test window
is scored in **one pass**, after every choice has been made, and `score_test_once` refuses a second call in
the same process. The numbers it returns are then asserted equal to `recorded_forecast.json`: a mismatch would
mean the window had been scored again under different choices, which is the one thing the protocol forbids.

In [ ]:
runs = fc.run_models(parts, corpus, space)     # the only choice made here is the ridge penalty, on 2015-2019
valid = fc.validation_metrics(runs, parts)
print(f"{'model':<44}{'validation RMSE':>17}{'validation MAE':>16}")
for key, m_ in valid.items():
    print(f"{runs[key].name:<44}{m_['rmse']:>17.4f}{m_['mae']:>16.4f}")
for key in ("tfidf_ridge_alone", "ols5_plus_tfidf_ridge"):
    d_ = runs[key].detail
    print(f"   {runs[key].name}: alpha {d_['alpha_chosen_on_validation']:g} chosen on 2015-2019 from "
          f"{sorted(float(a) for a in d_['validation_rmse_by_alpha'])}, {d_['n_columns']} columns")

test_scores = fc.score_test_once(runs, parts)               # one pass, after every choice
scored_once = fc._TEST_SCORED["count"] == GATE["forecast_test_scored_times"]
verdict = fc.assert_matches_record(test_scores, places=GATE["forecast_record_decimals"])
try:
    fc.score_test_once(runs, parts)
    refused = False
except RuntimeError as exc:
    refused = True
    print(f"\na second scoring attempt is refused: {exc}")
print(f"test-window numbers equal recorded_forecast.json to {GATE['forecast_record_decimals']} places: "
      f"{all(verdict.values())} ({len(verdict)} models)")

gate("F3", scored_once and refused and all(verdict.values()),
     f"the test window was scored once, a second attempt raised, and all {len(verdict)} models match "
     f"the recorded file")

## F4. The answer, in three columns

`frozen file` is Session 2's published forecast scored as shipped. `re-estimated` is the same model form
refitted on the 1994-onward rows. `plus text` adds the text regressors. The gap between the first two columns
is the cost of the sample restriction and has nothing to do with text; reporting it in its own column is what
stops the sum of two changes being attributed to one of them.

In [ ]:
print(f"{'model':<24}{'frozen file':>20}{'re-estimated':>20}{'plus text':>20}")
print(f"{'':<24}{'RMSE    MAE':>20}{'RMSE    MAE':>20}{'RMSE    MAE':>20}")
for row in fc.three_column_table(frozen, test_scores):
    cells = []
    for key in ("frozen_file", "re_estimated", "plus_text"):
        cell = row[key]
        cells.append(f"{'--':>8}{'--':>8}" if cell is None
                     else f"{cell['rmse']:>8.4f}{cell['mae']:>8.4f}")
    print(f"{row['model']:<24}" + "".join(f"{c:>20}" for c in cells))

actual = parts["test"]["y10_next"].to_numpy()
err = lambda k: actual - runs[k].predictions["test"]
print(f"\n{'comparison':<32}{'DM-HLN':>9}{'p':>8}{'Clark-West':>13}{'p (one-sided)':>15}")
for label, small, large, nested in (("OLS-5 vs OLS-5 + text", "ols5", "ols5_plus_text", True),
                                    ("OLS-5 vs OLS-5 + TF-IDF ridge", "ols5", "ols5_plus_tfidf_ridge", True),
                                    ("random walk vs text alone", "random_walk", "text_alone", False),
                                    ("random walk vs OLS-5", "random_walk", "ols5", False)):
    dm = fc.diebold_mariano(err(small), err(large))
    if nested:
        cw = fc.clark_west(actual, runs[small].predictions["test"], runs[large].predictions["test"])
        print(f"{label:<32}{dm['statistic']:>9.2f}{dm['p_value']:>8.3f}{cw['statistic']:>13.2f}"
              f"{cw['p_value_one_sided']:>15.3f}")
    else:
        print(f"{label:<32}{dm['statistic']:>9.2f}{dm['p_value']:>8.3f}{'—':>13}{'—':>15}")
print("\nDiebold-Mariano is two-sided and a NEGATIVE statistic favours the first model; the Harvey-Leybourne-")
print("Newbold correction matters at n = 79, where the uncorrected normal over-rejects. Clark-West is")
print("one-sided and a POSITIVE statistic favours the larger model: under the null that the extra regressors")
print("are zero in population, the larger model still has to estimate them and is EXPECTED to lose, which is")
print("what makes DM mis-sized for a nested pair. Both are reported for every nested pair; neither alone.")

In [ ]:
order = [("random_walk", MUTED, "-"), ("ols5", TEAL, "-"), ("text_alone", TEAL, "--"),
         ("ols5_plus_text", RED, "-")]
plt.figure(figsize=(7, 3.2))
for key, colour, style in order:
    plt.plot(test_scores[key]["cumulative_squared_error"], color=colour, ls=style, lw=1.1,
             label=f"{runs[key].name} ({test_scores[key]['rmse']:.4f})")
plt.xlabel("test month, 2020-01 onward"); plt.ylabel("cumulative squared error")
plt.legend(frameon=False, fontsize=7.5); plt.tight_layout(); plt.show()
print("A gap produced entirely by one month is a different finding from one that accumulates steadily, and")
print("only the path shows which. Counting the months in which each model stands ahead of the random walk:")
walk = np.array(test_scores["random_walk"]["cumulative_squared_error"])
for key, _, _ in order[1:]:
    ahead = int((np.array(test_scores[key]["cumulative_squared_error"]) - walk < 0).sum())
    print(f"   {runs[key].name:<44}{ahead:>3} of {len(walk)} months")
print("OLS-5 plus text is behind from the first month and never draws level. Nothing here rests on March 2020.")

In [ ]:
llm = fc.llm_column(frame)
print("the LLM column, beside the structured ones and never inside them (protocol section 8.1)")
print(f"   {llm['n_scored']} statements scored, variants recorded: {llm['variants_present']}, "
      f"variant A only enters a forecast")
print(f"   coverage of the estimation sample: {100 * llm['coverage_of_estimation_sample']:.0f}%")
print(f"   pooled into model selection: {llm['pooled_into_model_selection']}")
print(f"   test RMSE {llm['test_metrics']['rmse']:.4f}, MAE {llm['test_metrics']['mae']:.4f}   "
      f"against the random walk's {test_scores['random_walk']['rmse']:.4f}")
print(f"   coefficients: " + ", ".join(f"{k} {v:+.4f}" for k, v in llm["coefficients"].items()))
print("\nNo model choice and no penalty in this block depended on that column. It is reported because it was")
print("recorded, and it is outside the comparison because the protocol put it there before it was scored.")

## F5. Reported as found

**Text does not help.** On the 79 test months from 2020-01, the random walk is the best forecast in the table;
adding the three text regressors to OLS-5 makes it substantially worse; the TF-IDF ridge does the same; text
alone beats OLS-5 but not the random walk, which is a statement about OLS-5, not about text. Clark–West — the
statistic built to give the larger model the benefit of the doubt — still favours the smaller one.

Every gate in this block passed. That is the design working, not the design failing: the seven gates are on
procedure, and the protocol said in advance that this outcome passes all of them. A block whose gates could
only pass if text helped would be a block that could not report this result at all.

Three honest caveats, in the direction that weakens the finding rather than the claim:

- The window is 79 months and opens in January 2020, so a single monetary regime — the pandemic, the
  inflation surge and the tightening that answered it — is doing a great deal of the work. The cumulative
  path rules out one *month* driving the result; it does not rule out one *era*.
- The text features are three numbers — a seed-axis stance, a novelty index, and months since the last
  statement — plus a TF-IDF ridge. A richer representation is not ruled out by anything here.
- The comparator is strong. On this window the random walk beats every structured model Session 2 fitted, so
  "text does not beat the bar" is partly a statement about how high the bar happens to sit.

What the block does establish is narrower and firmer: under a design fixed and hashed in advance, with the
sample restriction reported in its own column and the test window scored once, this text signal does not
improve this forecast.

# Your turn

**1. Chapter 13, exercise 5 — the eight FOMC sentences, on this lab's path.** The chapter's version asks for
FinBERT through `transformers`, which this offline machine does not carry, so run the comparison on the two
instruments the notebook does have. The eight sentences are printed as the chapter prints them, in the cell
below. (a) Score each one with the chapter's dictionary: positive words {robust, strong, appropriate}, negative
{softened, elevated, risks, tightening}, the score being the sum of the positive terms' TF-IDF weights minus
the sum of the negative terms', with the weights taken from **this lab's** training-window model (`model`), so
the IDF comes from 149 FOMC statements rather than from the eight sentences. (b) Score the same eight with
`tx.signal_decision_sentence` and with the block B hawk–dove axis. (c) Correlate the three. Where do they
disagree most? (d) Sentence 4 — *"ongoing increases in the target range will be appropriate"* — carries the
positive word "appropriate" and is a forward-looking hawkish signal. Sentiment and stance are not the same
dimension. Which of your three scores gets it right, and what would you have to build to get it right on
purpose rather than by luck?

**2. Chapter 13, exercise 6 — the two-model consistency diagnostic, as printed.** (a) Run
`book_code/ch13/yin_vu_persico_kappa.py` (reach it with `me.book_module("yin_vu_persico_kappa", chapter="ch13")`)
and report the mean, standard deviation and ratio of the two score sets, their Cohen κ computed from the
definition, and the Landis–Koch band it falls in. (b) Modify the script to introduce a controllable drift
parameter $\rho \in [0,1]$, with $\rho = 0$ two models that agree exactly and $\rho = 1$ two that are
independent; plot $\kappa(\rho)$, and read off the $\rho$ implied by the lowest pairwise agreement Yin, Vu and
Persico report, $\kappa = 0.36$. (c) Put each score set into a simplified difference-in-differences,
$y_{it} = \alpha + \beta\,\mathrm{Post}_t \cdot \mathrm{Exposure}_i + u_{it}$, report the two $\hat\beta$ and
their ratio, and — if the signs differ — write the minimum disclosure list that would have to accompany the
paper.

**3. Chapter 14, programming items 1–3, as printed.** (1★) Run `human_llm_kappa.py` and reproduce the overall
and per-class κ. Then move the rare class's share from 0.12 down to 0.03 and plot both κ curves against it: at
what share does the overall figure stop reflecting the failure of that class, and what does that imply for
reporting standards? (2★★) Run `bias_correction.py` and confirm the naive and corrected estimators' bias and
interval widths. Then (a) set the size-related systematic term in the measurement error to zero and watch the
naive estimator's bias, and (b) drop $c_i$ from the calibration equation and check whether the corrected
estimator is still approximately unbiased — and say why. (3★★) Run `prompt_permutation_test.py` and reproduce
the three presentation designs. Then, at a fixed total call budget, does spending on more items or on more
orderings per item reduce bias, and does it reduce variance? Give a design recommendation for a four-option
labelling task with 2,000 calls, and state the condition under which your recommendation changes.

**4. A fifth signal: a dictionary uncertainty index.** The four signals in this notebook are the TF-IDF change
series, the embedding hawk–dove axis, the decision-sentence regex, and the recorded language-model stance.
Build a fifth in the Baker–Bloom–Davis manner: count the uncertainty words {uncertain, uncertainty, risks,
downside, volatility, unusually, strains, turbulence} per statement, divide by the statement's token count, and
push the series through `tx.validate_signal` on the same subsequent policy path. Expect it to fail the sign
test, and say precisely what it is measuring that stance is not — then ask whether it should ever have been
asked to separate hiking from cutting cycles in the first place.

Chapter 13's exercises 2, 3 and 4 need files or packages this machine does not carry — `gensim` with
`code/ch13/earnings_calls_sample.txt`, `sentence-transformers` with `analyst_reports_sample.txt`, and `jieba`
with two PBoC reports downloaded from the web — so they are worked on a connected machine rather than here.

In [ ]:
CHAPTER13_EX5 = [
    "The Committee decided to raise the target range for the federal funds rate.",
    "Inflation remains elevated, reflecting supply and demand imbalances.",
    "Job gains have been robust in recent months.",
    "The Committee anticipates that ongoing increases in the target range will be appropriate.",
    "The pace of future increases will take into account cumulative tightening.",
    "Recent indicators suggest that spending and production have softened.",
    "The Committee is highly attentive to inflation risks.",
    "The labor market remains strong.",
]
POSITIVE = ("robust", "strong", "appropriate")
NEGATIVE = ("softened", "elevated", "risks", "tightening")

probe = [tx.Statement(date=f"2022-01-{i + 1:02d}", text=s, file="chapter13-exercise-5")
         for i, s in enumerate(CHAPTER13_EX5)]
W = model.transform(probe)                     # this lab's IDF, from the 149 training statements
for i, sentence in enumerate(CHAPTER13_EX5):
    pos = sum(W[i, model.column[w]] for w in POSITIVE if w in model.column)
    neg = sum(W[i, model.column[w]] for w in NEGATIVE if w in model.column)
    print(f"{i + 1}. dictionary {pos - neg:+.4f}   {sentence}")
print("\nStart here. Then add the other two scores, and write down what sentence 4 does to each of them.")

---
**What the gates were.** A: the corpus hashes and the line at 2014-12-31; the merges learned on that window; a topic model fitted on the same window whose top words reproduce the recorded run;
an IDF computed from 149 documents. B: word vectors from the same 149, and both validation verdicts equal to
the recorded run — the one that fails and the one that passes, held to the same bound. C: the chapter's
printed arithmetic; five file hashes, a parameter count and a state dictionary; a causal mask that is exactly
zero above the diagonal; a held-out split by whole statement with every character accounted for. D: four
provenance fields on every one of 389 replies, one prompt hash shared by A and D, and the same 48 statements
under every variant. E: 48 digits printed in chapter 14, reproduced by the chapter's own code at the chapter's
own precision. F: two file hashes, three frozen baselines to four decimals, splits of 250 / 60 / 79, an unfilled
gap at 2025-11-30, nothing after 2014-12-31 inside a fitted object, and a test window scored once that refuses
to be scored again.

None of them is a threshold on a result. Every one of them would pass unchanged if block B's axis had
validated, if block D's repeat had been the steadier of the two, and if block F's text had won.

In [ ]:
print(f"{len(GATES)} gates, {sum(ok for _, ok in GATES)} passing:")
print("   " + "  ".join(name for name, _ in GATES))
print(f"notebook run time: {time.time() - T_START:.0f} s")